Sheet 0: Setup & Orientation
========================
**Authors: Polina Tsvilodub, Amir Mohammadpour**

This notebook covers the practical setup for the course and a first orientation to what a language model computes.

The goals of this session are:

* completing the Python environment setup for all practical exercises throughout the semester,
* verifying that key packages run correctly on your machine or on Colab,
* familiarising yourself with the coding conventions used throughout the course,
* observing the computations a language model performs, as direct motivation for what the following sessions build.

**Please attempt the installation section (below) ahead of the first tutorial session**, ideally while you have a stable internet connection. This way, any installation problems can be addressed during the session itself.

## Installing requirements

Throughout the semester, we will use Python, PyTorch, and a number of supporting packages for both in-tutorial exercises and homework assignments. All exercises require you to execute Python code yourself.

Setup can be done either on your own machine or via [Google Colab](https://colab.research.google.com/). Colab can be accessed by clicking the Colab icon at the top of the notebook page. Depending on your choice, follow the respective steps below.

Working with language models is compute-intensive: model weights require substantial disk space and, for training, access to a GPU. For most purposes in this course, **Colab is the recommended option** unless you have a performant GPU on your own machine. Two additional compute resources are also available: [bwJupyter](https://www.bwjupyter.de/), a server provided by the state of Baden-Württemberg, and a GPU API provided by NVIDIA. Details below.

### Colab

Colab is a free platform provided by Google that requires only a Google account. It provides limited access to GPU computation, which will be necessary for later sessions.

To enable GPU access: navigate to **Runtime > Change runtime type > GPU > Save**. Please be mindful of resource usage: Colab monitors GPU activity and may restrict access temporarily if usage is high.

Colab includes Python and many standard packages by default. The more specific packages required for this course must be installed manually, and must be reinstalled each time a new Colab runtime is started. To verify that your Colab environment is set up correctly, open this notebook in Colab (see above), uncomment, and run the following cell:

In [ ]:
# !pip install torch transformers matplotlib numpy

Alternatively, you can install from a requirements file by running `!pip install -r requirements.txt` once the file is available in your working directory.

### Local installation

Running exercises locally is a more advanced option. If you take this route, we strongly recommend creating a dedicated virtual environment (e.g., with Conda) before installing any packages. Check whether your GPU supports CUDA, MPS (Apple Silicon), or ROCm; if not, Colab can be used for GPU-dependent tasks while lighter exercises run locally.

Steps:
* Install Python >= 3.10
* Create and activate a virtual environment (recommended)
* If you have a GPU supported by deep learning frameworks, consult [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/) to select the correct PyTorch version
* Install the required packages: `pip install torch transformers matplotlib numpy`

### bwJupyter

bwJupyter is a compute service provided by the state of Baden-Württemberg for educational use. It hosts a Jupyter server with GPU access, though available resources per user are limited (2 GB disk, 8 GB RAM, ~6 GB GPU memory). We will use this service selectively.

Please attempt to log in with your **university email** at [hub.bwjupyter.de](https://hub.bwjupyter.de/hub/login?next=%2Fhub%2F) before the first session.

### NVIDIA API

NVIDIA hosts an API providing inference access to a range of state-of-the-art language models. This allows querying pretrained models without downloading weights, which can be useful for later sessions focused on prompting and evaluation. Note that API access does not expose internal model representations or token probabilities, and does not support training.

Sign-up and usage instructions are available [here](https://maxschmaltz.github.io//Course-LLM-based-Assistants/infos/llm_inference_guide/README.html).

## Verifying requirement installation

Run the cells below to confirm that the core packages are installed and working. If errors occur that you cannot resolve before the tutorial, flag them at the start of the session.

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import numpy as np

print('torch:', torch.__version__)

/opt/anaconda3/envs/lmprocessing/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.8.0


In [2]:
# confirm computation device
# on Colab with GPU enabled, or with a local CUDA GPU: 'cuda'
# on Apple Silicon (M1 or later): 'mps'
# otherwise: 'cpu'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

Device: mps


In [3]:
# basic tensor operations

x = torch.rand(5, 3).to(device)
y = torch.ones(5, 3).to(device)

print('x shape:', x.shape)
print('x @ y.T shape:', (x @ y.T).shape)  # matrix multiply: (5,3) x (3,5) -> (5,5)

x shape: torch.Size([5, 3])
x @ y.T shape: torch.Size([5, 5])


## Best practices for writing code

Throughout the course, submitted code is expected to be clean, readable, and documented. The following conventions apply:

* **PEP 8**: the standard Python style guide. It covers naming conventions, indentation, line length, and more. An overview is available [here](https://pep8.org/). Formatters such as *Black* or *Ruff* can enforce these conventions automatically and integrate as extensions in VS Code.
* **Docstrings**: every function should be directly followed by a docstring specifying its purpose, arguments, and return values. The course uses [numpydoc](https://numpydoc.readthedocs.io/en/latest/format.html#docstring-standard) style.

The distinction matters for practical reasons: code you write in week 2 may be read by you or a collaborator in week 10. Structure it accordingly.

In [4]:
# example: inadequate formatting and documentation
def add(a,b):
    """a+b"""
    return a+b


# example: PEP 8 formatting with a proper docstring
def add(a, b):
    """
    Add two numbers.

    Args
    ----
    a : int
        First number.
    b : int
        Second number.

    Returns
    -------
    int
        Sum of a and b.
    """
    return a + b

## What does a language model compute?

Before building anything from scratch, it is worth establishing precisely what a language model does at the level of computation. This section uses GPT-2 — a small but fully functional autoregressive language model — as a black box: the model is loaded from a pretrained checkpoint and used to perform three operations. No implementation details are explained here; those are reconstructed from first principles beginning with the next notebook. The goal is to make concrete the quantities that matter — the probability distribution over the next token, the surprisal of individual tokens in context, and the connection between those probabilities and text generation.

In [5]:
# load GPT-2 and its tokenizer
# the first run will download the weights (~500 MB); subsequent runs load from cache

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

Parameters: 124,439,808


### Next-token prediction

A language model defines a conditional probability distribution: given a sequence of tokens $x_1, x_2, \ldots, x_t$, the model assigns a probability to every possible continuation $x_{t+1}$. In practice, GPT-2 produces a vector of unnormalized scores — **logits** — over its vocabulary of approximately 50,000 tokens. Passing these through the softmax function converts them to a valid probability distribution. The cell below extracts this distribution at the final position of a prompt and displays the five most probable continuations.

In [6]:
prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    logits = model(**inputs).logits  # shape: (1, seq_len, vocab_size)

# distribution over the next token: take logits at the last position
next_token_logits = logits[0, -1, :]
probs = F.softmax(next_token_logits, dim=-1)

top5 = torch.topk(probs, 5)
print(f"Prompt: '{prompt}'\n")
print(f"{'Token':<20} {'Probability':>12}")
print('-' * 34)
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.decode(idx)
    print(f"{repr(token):<20} {prob.item():>12.4f}")

Prompt: 'The capital of France is'

Token                 Probability
----------------------------------
' the'                     0.0846
' now'                     0.0479
' a'                       0.0462
' France'                  0.0324
' Paris'                   0.0322


> <strong><span style="color:#D83D2B;">Exercise 0.1 &mdash; Distribution shape under uncertainty</span></strong>
>
> Run the next-token prediction cell above with each of the following prompts in turn, and examine how the top-5 output changes:
>
> 1. `'The bank by the'`
> 2. `'The gzorbax is a'`
>
> For each prompt, describe the shape of the top-5 distribution: are the probability values concentrated on a few tokens, or spread across many? What does the difference between the two cases tell you about what GPT-2 has and has not seen during training?
>
> As a follow-up: the **temperature** parameter in generation scales the logits before the softmax. Based on what you observe here, why might it be useful to have control over the sharpness of this distribution?

### Surprisal

Probability is an inconvenient unit: the probability assigned to a specific token from a large vocabulary is typically very small, and meaningful differences between model behaviours are compressed into a narrow range near zero. A more informative quantity is **surprisal**: the negative log-probability of a token given its context, measured in bits.

$$s(x_{t+1} \mid x_1, \ldots, x_t) = -\log_2 \, p(x_{t+1} \mid x_1, \ldots, x_t)$$

Surprisal is high when the model assigns low probability to the token that actually occurred, and low when the continuation was anticipated. It is the per-token quantity whose mean across a sequence the training objective — cross-entropy loss — directly minimises. The two sentences below share an identical two-word prefix but diverge at a structurally ambiguous point; the model's surprisal values reflect how much each continuation was expected given that shared context.

In [7]:
def token_surprisals(sentence, model, tokenizer):
    """
    Compute the surprisal (in bits) of each token given all preceding tokens.

    Args
    ----
    sentence : str
        Input sentence.
    model : transformers.PreTrainedModel
        Pretrained autoregressive language model.
    tokenizer : transformers.PreTrainedTokenizer
        Tokenizer corresponding to the model.

    Returns
    -------
    list of tuple[str, float]
        (token_string, surprisal_in_bits) for each token from position 1 onward.
        The first token has no context and is excluded.
    """
    ids = tokenizer.encode(sentence, return_tensors='pt')  # (1, seq_len)
    with torch.no_grad():
        logits = model(ids).logits  # (1, seq_len, vocab_size)
    log_probs = F.log_softmax(logits, dim=-1)  # (1, seq_len, vocab_size)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    log2 = torch.log(torch.tensor(2.0)).item()
    results = []
    for i in range(len(ids[0]) - 1):
        # log_probs[0, i, :] is the distribution over the token at position i+1
        token_id = ids[0, i + 1].item()
        log_p = log_probs[0, i, token_id].item()
        results.append((tokens[i + 1], -log_p / log2))
    return results


sentences = [
    'The bank by the river flooded.',
    'The bank approved the loan.',
]

for sentence in sentences:
    result = token_surprisals(sentence, model, tokenizer)
    print(f'\n{sentence}')
    print(f"  {'Token':<20} {'Surprisal (bits)':>18}")
    print('  ' + '-' * 40)
    for token, s in result:
        print(f"  {repr(token):<20} {s:>18.2f}")


The bank by the river flooded.
  Token                  Surprisal (bits)
  ----------------------------------------
  'Ġbank'                           12.56
  'Ġby'                             12.02
  'Ġthe'                             3.13
  'Ġriver'                           7.71
  'Ġflooded'                        12.17
  '.'                                3.49

The bank approved the loan.
  Token                  Surprisal (bits)
  ----------------------------------------
  'Ġbank'                           12.56
  'Ġapproved'                       12.62
  'Ġthe'                             1.16
  'Ġloan'                            3.92
  '.'                                4.87


> <strong><span style="color:#D83D2B;">Exercise 0.2 &mdash; Reading surprisal values</span></strong>
>
> Inspect the surprisal output for the two sentences above and answer the following:
>
> 1. Do the surprisal values for `'The'` and `'bank'` differ between the two sentences? Should they? In one sentence, explain what this implies about what information the model has encoded at the point of ambiguity.
>
> 2. Using the values printed above, compute the **mean surprisal** across all tokens for each sentence. This quantity — mean surprisal in bits — is directly proportional to per-token cross-entropy, which is what the model is trained to minimise. Given that GPT-2 was trained predominantly on web text, what does the difference in mean surprisal between the two sentences suggest about which type of continuation the training corpus rewarded more strongly?
>
> *You do not need to write code for this exercise. Inspect the printed output and reason from it.*

### Generation

Once a model can assign probabilities to every possible continuation, text generation follows directly: select a next token according to the model's distribution, append it to the context, and repeat. The simplest strategy is **greedy decoding**, which always selects the token with the highest probability. This is deterministic but tends toward repetitive output, because the most probable continuation of a sequence is often also the most probable continuation of that continuation.

**Temperature sampling** instead draws tokens from the full distribution after rescaling the logits by a temperature parameter $T$:

$$p_T(x) \propto \exp\!\left(\frac{\text{logit}(x)}{T}\right)$$

Values of $T < 1$ sharpen the distribution, concentrating probability on high-scoring tokens. Values of $T > 1$ flatten it, making lower-probability tokens more reachable. The cell below runs both strategies on the same prompt; re-running it will produce different sampled outputs.

In [8]:
prompt = 'Language models assign probabilities to'
inputs = tokenizer(prompt, return_tensors='pt')

# greedy: deterministic, always selects the most probable next token
greedy_output = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False,
)

# temperature sampling: stochastic, draws from the rescaled distribution
sampled_output = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.8,
)

print('Greedy decoding:')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

print('\nTemperature sampling (T = 0.8):')
print(tokenizer.decode(sampled_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Greedy decoding:
Language models assign probabilities to the probability of a given event. The probability of a given event is the probability of the event being observed.

The probability of a given event

Temperature sampling (T = 0.8):
Language models assign probabilities to certain information elements. The probability of having a certain element is a function of the probability of having the given element. More about probability distributions and probability distributions


> <strong><span style="color:#D83D2B;">Exercise 0.3 &mdash; Greedy decoding and global optimality</span></strong>
>
> Greedy decoding selects the most probable token at each step. It does not, in general, produce the most probable complete sequence. Construct a small counterexample by hand to demonstrate this.
>
> Specifically: suppose a vocabulary of four tokens $\{A, B, C, D\}$ and a two-step generation process. Invent conditional probability values (they need not be realistic, only internally consistent) such that greedy decoding produces a two-token sequence with lower joint probability than some other sequence it did not choose.
>
> Write out the probability tables and show the calculation. Then answer: if greedy decoding does not find the highest-probability sequence, what property does it guarantee, and why might it still be used in practice?
>
> *No code required. This is a pencil-and-paper exercise.*

The model used above has 124 million parameters. The probabilities it assigns are the output of a function that was optimised, over a large corpus of text, to minimise the average surprisal of each next token given its context. That optimisation required computing the gradient of the loss with respect to every one of those parameters — at every training step, across billions of token predictions. Understanding how those gradients are computed automatically is the subject of the next notebook.